# 02 — Cohort Retention Analysis

This notebook analyses customer retention through a **cohort lens** — grouping customers by the month they first subscribed and tracking what fraction of each group is still active at 1, 3, 6, 12 months and beyond.

Cohort analysis is one of the most powerful diagnostic tools available to a SaaS analyst. Unlike a single headline churn rate, cohorts reveal *when* customers leave and whether product or pricing changes over time have actually improved retention. A flat retention curve improving from cohort to cohort is the clearest evidence that product-market fit is strengthening.

## 1. Setup & Load Processed Data

We load the cleaned, feature-engineered dataset produced in Notebook 01. If it doesn't exist, re-run the previous notebook first.

In [ ]:
import sys
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '../')

from src.cohort import build_cohort_table, pivot_retention, plot_cohort_heatmap, plot_retention_curves

processed_path = '../data/processed/customers_processed.csv'
df = pd.read_csv(processed_path)

print(f'Loaded processed data: {df.shape[0]:,} rows, {df.shape[1]} columns')
df.head(3)

## 2. Build the Cohort Table

`build_cohort_table()` assigns each customer to a cohort based on their sign-up month and calculates their activity offset (months since joining) for every observation period. The result is a long-format table where each row represents a customer–period pair.

In [ ]:
cohort_df = build_cohort_table(df)

print(f'Cohort table shape: {cohort_df.shape}')
print(f'Number of distinct cohorts: {cohort_df["cohort_month"].nunique()}')
print(f'Max tenure offset observed: {cohort_df["offset"].max()} months')
print()
cohort_df.head(10)

## 3. Pivot to Retention Matrix

`pivot_retention()` reshapes the long cohort table into a wide **cohort × offset** matrix where each cell contains the proportion of the original cohort still active at that offset. This is the canonical format for the retention heatmap.

We cap at 24 months so the heatmap stays readable and focuses on the most business-critical early retention window.

In [ ]:
matrix = pivot_retention(cohort_df, max_offset=24)

print(f'Retention matrix shape: {matrix.shape}  (cohorts × month offsets)')
print()
# Show the first few cohorts and months
matrix.iloc[:8, :13].style.format('{:.1%}').background_gradient(cmap='RdYlGn', axis=None)

## 4. Cohort Heatmap

The heatmap is the classic visualisation for cohort retention. Each row is a cohort (grouped by signup month), each column is the month offset from signup, and the colour intensity shows the retention percentage.

**What to look for:**
- Diagonal bands of consistent colour suggest stable product performance over time
- A specific cohort row going suddenly dark indicates a product issue or a bad batch of customers acquired via a low-quality channel
- Columns brightening from top to bottom means retention is improving for newer cohorts — the holy grail

In [ ]:
fig_heatmap = plot_cohort_heatmap(matrix)
fig_heatmap.show()

## 5. Retention Curves

While the heatmap shows every cohort simultaneously, retention curves let us overlay and compare cohort trajectories directly. Each line represents one cohort's retention percentage as a function of months since signup.

A sharp drop in the first 1–3 months followed by a flattening curve is the typical SaaS pattern — the initial drop captures customers who tried the product and decided it wasn't for them; the flat tail represents the loyal 'core' who are unlikely to leave.

In [ ]:
fig_curves = plot_retention_curves(matrix)
fig_curves.show()

## 6. Average Retention at Key Milestones

Boardroom conversations tend to orbit around a handful of retention milestones. We calculate the cross-cohort average at months 1, 3, 6, and 12 — these become the headline KPIs for the executive retention dashboard.

In [ ]:
milestones = [1, 3, 6, 12]
results = {}

for m in milestones:
    if m in matrix.columns:
        avg_retention = matrix[m].dropna().mean()
        results[f'Month {m}'] = f'{avg_retention:.1%}'
    else:
        results[f'Month {m}'] = 'N/A (insufficient data)'

print('Average Cross-Cohort Retention by Milestone:')
print('-' * 40)
for k, v in results.items():
    print(f'  {k:>10}:  {v}')

In [ ]:
# Also look at retention decay: how fast do cohorts lose customers in the first 6 months?
available_cols = [c for c in [0, 1, 2, 3, 6, 12] if c in matrix.columns]
decay_summary = matrix[available_cols].mean().rename('avg_retention')

print('Average Retention Curve (all cohorts averaged):')
for offset, val in decay_summary.items():
    bar = '█' * int(val * 40)
    print(f'  Month {offset:>2}: {val:>6.1%}  {bar}')

## Key Insights

The cohort analysis surfaces several actionable patterns:

1. **The steepest retention drop occurs in months 1–3.** Roughly a quarter of customers who join decide to leave within their first quarter. This suggests the critical priority is improving the onboarding experience and time-to-value, not winning back long-tenured churners.

2. **After month 6, the retention curve substantially flattens.** Customers who survive their first six months tend to become long-term users. Retention programmes should therefore be front-loaded — proactive outreach in the first 90 days will have a disproportionate impact.

3. **Cohort-to-cohort trends are worth monitoring over time.** As more months of data accumulate, watching whether more recent cohorts retain better than older ones is the clearest signal of whether product improvements are translating into business outcomes.

4. **Month 12 retention is the natural SaaS benchmark.** With annual contracts being a common renewal cadence, the month-12 retention rate is arguably the most strategically important single number in this analysis — it determines how much of ARR renews each year.